Datos de Clausura 2025

# Import necessary libraries

In [4]:
# Import libraries 
import pandas as pd
import requests
from scipy.optimize import minimize
import numpy as np

In [7]:
!pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 11.0 MB/s  0:00:01 eta 0:00:01


In [8]:
import pulp

# Import, clean and prepare data

For the proyect, I took the data from transfermarkt and the official website of the Liga MX. However, the variables are different and there's no score. So I made the scoring function based on the premier league function. 

Let's begin by importing the raw data.

In [42]:
raw_data = pd.read_excel('/Users/gustavo/Documents/03 Proyecto Liga MX/datos_completos.xlsx')
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0
1,Rodrigo Aguirre,URU,"DL,CC",América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1
2,Roberto Alvarado,MEX,"CC,DL",Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0


Some players have two positions. We'll stay with the first one for this project. 

In [43]:
raw_data['Posc'] = raw_data['Posc'].str.split(',').str[0].str.strip()

In [44]:
print(raw_data.columns)

Index(['Jugador', 'País', 'Posc', 'Equipo', 'Edad', 'Nacimiento',
       'PJ_tiempo_jugado', 'Titular_tiempo_jugado', 'Mín_tiempo_jugado',
       '90 s_tiempo_jugado', 'Gls._rendimiento', 'Ass_rendimiento',
       'G+A_rendimiento', 'G-TP_rendimiento', 'TP_rendimiento',
       'TPint_rendimiento', 'TA_rendimiento', 'TR_rendimiento',
       'Gls._por_90_mins', 'Ast_por_90_mins', 'G+A_por_90_mins',
       'G-TP_por_90_mins', 'G+A-TP_por_90_mins', 'Valor de mercado', 'Posc_PO',
       'Posc_DF', 'Posc_CC', 'Posc_DL'],
      dtype='str')


Vamos a cambiar los valores de posición por su equivalente en pulp. Donde el portero es 1, defensa 2, medio 3 y delantero 4. 

In [45]:
valores_unicos = raw_data["Posc"].unique()
print(valores_unicos)

['PO' 'DL' 'CC' 'DF']


In [46]:
# Mapeamos las posiciones a los valores numéricos
raw_data['position'] = raw_data['Posc'].map({'PO': 1, 'DF': 2, 'CC': 3, 'DL': 4})

Para el valor de cada jugador, pasamos la cifra a decimal de millón. Así es como está reportado su uso en pulp

In [47]:
raw_data["Valor de mercado_en milloes"] = (pd.to_numeric(raw_data["Valor de mercado"])/1_000_000).round(2)
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL,position,Valor de mercado_en milloes
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0,1,3.8
1,Rodrigo Aguirre,URU,DL,América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1,4,3.0
2,Roberto Alvarado,MEX,CC,Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1,3,7.5
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0,2,5.5
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0,3,6.5


## ICT score y puntos

En fantasy, se usa un ICT (Influence, Creativity, Threat) index. Evidentemente, esto no existe en la Liga MX. Sin embargo, podemos hacer un íncice sintético con los datos que tenemos. Esto permite hacer diferentes índices y compararlos. 

In [48]:
raw_data["Influence"] = (
      5 * pd.to_numeric(raw_data["Gls._rendimiento"], errors="coerce")
    + 3 * pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 0.01 * pd.to_numeric(raw_data["Mín_tiempo_jugado"], errors="coerce")
)

raw_data["Creativity"] = (
      5 * pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 2 * pd.to_numeric(raw_data["Ast_por_90_mins"], errors="coerce")
)

raw_data["Threat"] = (
      5 * pd.to_numeric(raw_data["Gls._por_90_mins"], errors="coerce")
    + 2 * pd.to_numeric(raw_data["G+A_por_90_mins"], errors="coerce")
)

In [49]:
raw_data["ICT"] = (
    raw_data["Influence"]
    + raw_data["Creativity"]
    + raw_data["Threat"]
)

In [50]:
# Normalizamos con scikit-learn preprocessing
from sklearn.preprocessing import MinMaxScaler 
raw_data["ICT_norm"] = MinMaxScaler().fit_transform(raw_data[["ICT"]])

In [51]:
# Para los puntos 
raw_data["points"] = (
      6*pd.to_numeric(raw_data["Gls._rendimiento"], errors="coerce")
    + 4*pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 0.02*pd.to_numeric(raw_data["Mín_tiempo_jugado"], errors="coerce")
    - 1*pd.to_numeric(raw_data["TA_rendimiento"], errors="coerce")
    - 3*pd.to_numeric(raw_data["TR_rendimiento"], errors="coerce")
)
raw_data['points_norm'] = MinMaxScaler().fit_transform(raw_data[["points"]])

Listo. Algunos jugadores tienen valores faltantes, como la muestra es pequeña, rellenaremos con la media. 

In [52]:
raw_data["ICT_norm"] = raw_data["ICT_norm"].fillna(raw_data["ICT_norm"].mean())
raw_data["points_norm"] = raw_data["points_norm"].fillna(raw_data["points_norm"].mean())

In [53]:
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL,position,Valor de mercado_en milloes,Influence,Creativity,Threat,ICT,ICT_norm,points,points_norm
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0,1,3.8,NaN,0.00,0.00,NaN,0.233223,NaN,0.245113
1,Rodrigo Aguirre,URU,DL,América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1,4,3.0,34.59,5.28,5.04,44.91,0.259907,41.18,0.232550
2,Roberto Alvarado,MEX,CC,Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1,3,7.5,75.76,30.60,3.47,109.83,0.643300,104.52,0.597328
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0,2,5.5,12.73,5.18,0.18,18.09,0.101518,19.46,0.107464
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0,3,6.5,14.91,0.00,0.63,15.54,0.086458,23.82,0.132573


# Implementación PULP

Nos basamos en estas variables de implementación. Ojo que son listas. 

In [ ]:
'''
# Extract player details into lists
players_name=list(df['web_name'])
points=list(df['points'])
price=list(df['price'])
positions=list(df['position'])
team=list(df['team'])
gkp=list(df['gkp'])
defender=list(df['def'])
mid=list(df['mid'])
fwd=list(df['fwd'])
num_players=len(df)
''' 

Objective Function

f(x) = sum(points)

Constraints:
- Max budget = 100.0
- Exactly 2 gks, 5 defenders, 5 midfielders, 3 forwards
- Max 3 players per team


In [54]:
players_name = list(raw_data['Jugador'])
points=list(raw_data['points_norm'])
price=list(raw_data['Valor de mercado_en milloes'])
positions=list(raw_data['position'])
team=list(raw_data['Equipo'])
num_players=len(raw_data)

In [56]:
# Define fpl budget to spend
total_budget=100

In [66]:
!pip install highspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 10.4 MB/s  0:00:00 eta 0:00:01


In [ ]:
# 1. Crear el problema de maximización
prob = pulp.LpProblem("Fantasy_Soccer_Optimizer", pulp.LpMaximize)

# 2. Crear las variables de decisión binary (0 si no se elige, 1 si se elige)
player_vars = [
    pulp.LpVariable(f"player_{i}", cat="Binary") for i in range(num_players)
]

# 3. Función Objetivo: Maximizar los puntos normalizados
prob += (
    pulp.lpSum([points[i] * player_vars[i] for i in range(num_players)]),
    "Total_Points",
)

# 4. Restricciones:

# Restricción A: Presupuesto Máximo (Max Budget = 100.0)
prob += (
    pulp.lpSum([price[i] * player_vars[i] for i in range(num_players)]) <= 100.0,
    "Max_Budget",
)

# Restricción B: Número exacto de jugadores por posición
# 1 GK (PO), 5 DEF (DF), 5 MID (CC), 3 FWD (DL)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 1
    ])
    == 1,
    "Exactly_1_GK",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 2
    ])
    == 5,
    "Exactly_5_DEF",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 3
    ])
    == 5,
    "Exactly_5_MID",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 4
    ])
    == 3,
    "Exactly_3_FWD",
)

# Restricción C: Máximo 3 jugadores por equipo
# No elijas más de 3 jugadores pertenecientes al mismo club en tu plantilla
unique_teams = set(team)
for t in unique_teams:
    prob += (
        pulp.lpSum([
            player_vars[i] for i in range(num_players) if team[i] == t
        ])
        <= 3,
        f"Max_3_players_{t}",
    )

# 5. Resolver el problema
'''
Aquí es probable que necesites especificar la ruta del solver CBC si no está en tu PATH. 
Asegúrate de tener instalado el solver CBC y de conocer su ubicación. 
En una de esas y necesitas, primero, instalar homebrew y luego instalar el solver CBC con el comando "brew install cbc".
Al instalar homebrew, no se te pase poner bien el path al final con:

echo >> /Users/_nombre user_/.zprofile
echo 'eval "$(/opt/homebrew/bin/brew shellenv zsh)"' >> /Users/_nombre user_/.zprofile
eval "$(/opt/homebrew/bin/brew shellenv zsh)" 

Por ejemplo, si estás usando Anaconda, la ruta podría ser algo como "/path/to/anaconda3/bin/cbc". 
Ajusta la ruta según tu instalación.
'''

solver = pulp.COIN_CMD(path="/opt/homebrew/bin/cbc", msg=False)
prob.solve(solver)

# 6. Extraer los resultados en un DataFrame
selected_indices = [
    i for i in range(num_players) if player_vars[i].varValue == 1.0
]
optimal_lineup = raw_data.iloc[selected_indices].copy()

print(f"Estado de la solución: {pulp.LpStatus[prob.status]}")
print(f"Puntos Totales Promedio: {pulp.value(prob.objective):.2f}")
print(
    f"Costo Total: {sum(price[i] for i in selected_indices):.2f}M (de 100.0M)"
)
print(f"Jugadores Seleccionados: {len(selected_indices)} (de 14 requeridos)")

# Mostrar la alineación seleccionada
optimal_lineup[
    ['Jugador', 'position', 'Equipo', 'Valor de mercado_en milloes', 'points_norm']
]

Estado de la solución: Optimal
Puntos Totales Promedio: 7.30
Costo Total: 81.80M (de 100.0M)
Jugadores Seleccionados: 14 (de 14 requeridos)


,Jugador,position,Equipo,Valor de mercado_en milloes,points_norm
2,Roberto Alvarado,3,Guadalajara,7.5,0.597328
13,Germán Berterame,4,Monterrey,7.0,0.568648
19,Diber Cambindo,4,Necaxa,4.0,0.683253
21,Sergio Canales,3,Monterrey,7.0,0.776088
33,Willer Ditta,2,Cruz Azul,6.0,0.245113
41,Jesús Gallardo,2,Toluca,3.8,0.411311
53,Joaquim,2,UANL,5.0,0.313177
61,Kevin Mier,1,Cruz Azul,8.0,0.245113
62,Alan Mozo,2,Guadalajara,5.5,0.245113
65,Agustín Oliveros,2,Necaxa,3.5,0.274476


In [77]:
equipo_ideal = optimal_lineup.sort_values('position').reset_index(drop=True)
equipo_ideal[['Jugador', 'position', 'Equipo', 'Valor de mercado_en milloes']]

,Jugador,position,Equipo,Valor de mercado_en milloes
0,Kevin Mier,1,Cruz Azul,8.0
1,Willer Ditta,2,Cruz Azul,6.0
2,Jesús Gallardo,2,Toluca,3.8
3,Joaquim,2,UANL,5.0
4,Alan Mozo,2,Guadalajara,5.5
5,Agustín Oliveros,2,Necaxa,3.5
6,Roberto Alvarado,3,Guadalajara,7.5
7,Sergio Canales,3,Monterrey,7.0
8,José Paradela,3,Necaxa,5.0
9,Carlos Rotondi,3,Cruz Azul,6.0


## Con capitán y banca

In [79]:
# 1. Crear el problema de maximización
prob = pulp.LpProblem("Fantasy_Soccer_15_Players_Optimizer", pulp.LpMaximize)

# 2. Definir las 3 variables de decisión por jugador (binarias)
starters = [
    pulp.LpVariable(f"starter_{i}", cat="Binary") for i in range(num_players)
]
bench = [pulp.LpVariable(f"bench_{i}", cat="Binary") for i in range(num_players)]
captain = [
    pulp.LpVariable(f"captain_{i}", cat="Binary") for i in range(num_players)
]

# 3. Función Objetivo: Maximizar puntos de titulares + bonificación de Capitán (+100% puntos)
# Puntos = (Puntos Titulares) + (Puntos Extra del Capitán)
prob += (
    pulp.lpSum([
        points[i] * starters[i] + points[i] * captain[i]
        for i in range(num_players)
    ]),
    "Total_Points_With_Captain",
)

# 4. RESTRICCIONES GENERALES DEL EQUIPO (15 Jugadores)

# Exactamente 11 titulares y 4 en banca
prob += pulp.lpSum(starters) == 11, "Exactly_11_Starters"
prob += pulp.lpSum(bench) == 4, "Exactly_4_Bench"

# Presupuesto Máximo de 100.0M considerando el costo total del plantel (15 jugadores)
prob += (
    pulp.lpSum([
        price[i] * (starters[i] + bench[i]) for i in range(num_players)
    ])
    <= 100.0,
    "Max_Budget_100M",
)

# Máximo 3 jugadores por equipo en la plantilla total de 15
unique_teams = set(team)
for t in unique_teams:
    prob += (
        pulp.lpSum([
            starters[i] + bench[i] for i in range(num_players) if team[i] == t
        ])
        <= 3,
        f"Max_3_players_{t}",
    )

# 5. RESTRICCIONES DE LÓGICA DE JUGADOR Y CAPITÁN

# Exactamente 1 capitán
prob += pulp.lpSum(captain) == 1, "Exactly_1_Captain"

for i in range(num_players):
    # El capitán DEBE ser un titular
    prob += starters[i] - captain[i] >= 0, f"Captain_Must_Be_Starter_{i}"

    # Un jugador NO puede ser titular y banca al mismo tiempo (máximo 1 rol)
    prob += starters[i] + bench[i] <= 1, f"Exclusive_Role_{i}"

# 6. RESTRICCIONES POR POSICIÓN (1=PO, 2=DF, 3=CC, 4=DL)

# Porteros (PO / 1): 1 titular, 2 en total (1 en banca)
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 1])
    == 1,
    "Starter_1_GK",
)
prob += (
    pulp.lpSum([
        starters[i] + bench[i] for i in range(num_players) if positions[i] == 1
    ])
    == 2,
    "Total_2_GK",
)

# Defensas (DF / 2): Entre 3 y 5 titulares, 5 en total
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 2])
    >= 3,
    "Min_3_DEF",
)
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 2])
    <= 5,
    "Max_5_DEF",
)
prob += (
    pulp.lpSum([
        starters[i] + bench[i] for i in range(num_players) if positions[i] == 2
    ])
    == 5,
    "Total_5_DEF",
)

# Mediocampistas (CC / 3): Entre 3 y 5 titulares, 5 en total
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 3])
    >= 3,
    "Min_3_MID",
)
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 3])
    <= 5,
    "Max_5_MID",
)
prob += (
    pulp.lpSum([
        starters[i] + bench[i] for i in range(num_players) if positions[i] == 3
    ])
    == 5,
    "Total_5_MID",
)

# Delanteros (DL / 4): Entre 1 y 3 titulares, 3 en total
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 4])
    >= 1,
    "Min_1_FWD",
)
prob += (
    pulp.lpSum([starters[i] for i in range(num_players) if positions[i] == 4])
    <= 3,
    "Max_3_FWD",
)
prob += (
    pulp.lpSum([
        starters[i] + bench[i] for i in range(num_players) if positions[i] == 4
    ])
    == 3,
    "Total_3_FWD",
)

# 7. Resolver con COIN_CMD
solver = pulp.COIN_CMD(path="/opt/homebrew/bin/cbc", msg=False)
prob.solve(solver)

# 8. Extraer resultados y mostrar la alineación estructurada
raw_data["is_starter"] = [starters[i].varValue for i in range(num_players)]
raw_data["is_bench"] = [bench[i].varValue for i in range(num_players)]
raw_data["is_captain"] = [captain[i].varValue for i in range(num_players)]

titulares_df = raw_data[raw_data["is_starter"] == 1].copy()
banca_df = raw_data[raw_data["is_bench"] == 1].copy()

print(f"Estado de la solución: {pulp.LpStatus[prob.status]}")
print(f"Puntos Totales Esperados (con Capitán): {pulp.value(prob.objective):.2f}\n")
print(f"Costo Total del Plantel: {sum(price[i] for i in range(num_players) if starters[i].varValue == 1 or bench[i].varValue == 1):.2f}M (de 100.0M)")  

print("--- 11 TITULARES ---")
print(
    titulares_df[[
        "Jugador",
        "position",
        "Equipo",
        "Valor de mercado_en milloes",
        "points_norm",
        "is_captain",
    ]]
)

print("\n--- 4 SUPLENTES ---")
print(
    banca_df[[
        "Jugador",
        "position",
        "Equipo",
        "Valor de mercado_en milloes",
        "points_norm",
    ]]
)

Estado de la solución: Optimal
Puntos Totales Esperados (con Capitán): 7.24

Costo Total del Plantel: 81.60M (de 100.0M)
--- 11 TITULARES ---
             Jugador  position       Equipo  Valor de mercado_en milloes  \
0     Carlos Acevedo         1       Santos                          3.8   
2   Roberto Alvarado         3  Guadalajara                          7.5   
19    Diber Cambindo         4       Necaxa                          4.0   
21    Sergio Canales         3    Monterrey                          7.0   
41    Jesús Gallardo         2       Toluca                          3.8   
53           Joaquim         2         UANL                          5.0   
65  Agustín Oliveros         2       Necaxa                          3.5   
66     José Paradela         3       Necaxa                          5.0   
67          Paulinho         4       Toluca                          8.0   
77    Carlos Rotondi         3    Cruz Azul                          6.0   
87       Alexis Vega  